# 문서 탐색 (EDA) — 제공 뉴스

벡터 DB 구성 전에 문서의 구성과 품질을 파악한다 (가이드 4-4).
청킹 방식, 전처리 규칙, 품질 필터(`indexer/quality.py`) 기준은 이 노트북에서 확인한 결과를 근거로 정하고 리포트 2장에 기술한다.

**실행 전**: 제공 CSV 를 `data/provided/` 에 넣고 `python -m collector provided` 로 적재한다.

> 출력에는 기사 본문 일부가 포함되므로, 커밋 전에 **Clear All Outputs** 로 지운다.

In [1]:
import json
import re
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import plotly.express as px
from IPython.display import display

from common.docstore import open_db
from common.schema import DocType
# 판정 규칙은 노트북에 다시 쓰지 않고 인덱서 코드에서 가져온다 → EDA 숫자 = 실제 인덱싱 결과
from indexer.quality import (
    CITY_PATTERN, DATELINE_PATTERN, DEFAULT_TOPIC_TERMS, NOT_PLACE, PROVINCE_PATTERNS, SEOUL_PATTERN,
    TITLE_MARKERS, QualityInput, assess, get_quality_config,
)

CONFIG = get_quality_config("v1")

px.defaults.template = "plotly_white"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]  # 범주 색상 고정 순서 (1~3번)

with open_db(read_only=True) as conn:
    news = pd.read_sql_query("SELECT * FROM documents WHERE doc_type = 'news'", conn)

news["extra"] = news["extra"].map(json.loads)
news["published"] = pd.to_datetime(news["published_at"], errors="coerce")
news["text"] = news["raw_text"].fillna("")
news["body_len"] = news["text"].str.len()
news["file_group"] = news["extra"].map(
    lambda e: " + ".join(sorted({"소상공인" if f.startswith("01_") else "지역화폐" for f in e["provided_files"]}))
)
print(f"뉴스 {len(news):,}건")


def show_samples(df, n, seed, width=150):
    """제목과 본문 앞부분을 출력한다. 수치만으로는 문서 성격을 알 수 없으므로 직접 읽는다."""
    for _, r in df.sample(min(n, len(df)), random_state=seed).iterrows():
        print(f"[{r['body_len']:,}자] {r['source']} | {r['title']}\n    {r['text'][:width]}")

뉴스 76,344건


## 1. 구성
- 파일(그룹)별, 수집 키워드별 건수
- 매체 분포, 매체명 결손·도메인 형태 복구 여부

In [2]:
display(news["file_group"].value_counts().rename("건수").to_frame())

# 묶음 파일의 복수 키워드 표기 형식은 원본마다 다를 수 있어 흔한 구분자를 모두 나눈다.
keywords = news["keyword"].fillna("").str.split(r"\s*[;,|/]\s*", regex=True).explode()
display(keywords[keywords != ""].value_counts().rename("건수").to_frame())

missing_source = news["extra"].map(lambda e: e["source_missing"])
print(f"매체 수 {news['source'].nunique():,} / 매체명 결손 {missing_source.mean():.1%}")
if missing_source.any():
    display(news[missing_source].groupby(news["published"].dt.to_period("M")).size().rename("결손 건수").tail(12).to_frame())
# 제공 측에서 URL·도메인으로 매체명을 복구했으므로, 매체명 자리에 도메인이 들어간 경우를 확인한다.
domain_like = news["source"].str.contains(r"\.(?:com|co\.kr|kr|net)\b", regex=True)
print(f"매체명이 도메인 형태인 기사: {domain_like.sum():,}건")
display(news.loc[domain_like, "source"].value_counts().head(10).rename("건수").to_frame())

top_sources = news["source"].value_counts().head(20).rename_axis("매체").reset_index(name="기사 수")
fig = px.bar(top_sources, x="기사 수", y="매체", orientation="h", title="매체별 기사 수 (상위 20)",
             color_discrete_sequence=SERIES[:1])
fig.update_yaxes(categoryorder="total ascending", title=None)
fig.update_layout(bargap=0.35)
fig.show()

,건수
file_group,
소상공인,39457
지역화폐,33894
소상공인 + 지역화폐,2993


,건수
keyword,
지역화폐,36887
골목상권,23968
상권 활성화,11344
전통시장 활성화,7003
소상공인 폐업,2135
소상공인 지원정책,381


매체 수 108 / 매체명 결손 0.0%
매체명이 도메인 형태인 기사: 0건


,건수
source,


## 2. 기간
- 발행일 분포, 특정 시기 편중 여부

In [3]:
print("기간:", news["published"].min(), "~", news["published"].max())
print("발행일 미상:", news["published"].isna().sum())

monthly = (
    news.dropna(subset=["published"])
    .assign(month=lambda d: d["published"].dt.to_period("M").dt.to_timestamp())
    .groupby(["month", "file_group"]).size().reset_index(name="기사 수")
)
fig = px.line(monthly, x="month", y="기사 수", color="file_group", title="월별 기사 수",
              color_discrete_sequence=SERIES, labels={"month": "", "file_group": "그룹"})
fig.update_traces(line_width=2)
fig.show()

기간: 2021-10-16 00:03:00 ~ 2026-10-16 15:01:00
발행일 미상: 0


## 3. 길이
- 본문 길이 분포, 과소·과다 문서 비율 → `QualityConfig.min_chars` 후보
- 짧은 문서와 가장 긴 문서를 직접 읽어 성격을 확인한다

In [4]:
print(news["body_len"].describe(percentiles=[.05, .1, .25, .5, .75, .9, .95, .99]).round(0))
for n in (100, 200, 300, 500):
    short = news["body_len"] < n
    print(f"본문 {n}자 미만: {short.sum():,}건 ({short.mean():.1%})")

q99 = news["body_len"].quantile(0.99)
fig = px.histogram(news[news["body_len"] <= q99], x="body_len", nbins=60, title="본문 길이 분포 (상위 1% 제외)",
                   labels={"body_len": "본문 길이(자)"}, color_discrete_sequence=SERIES[:1])
fig.update_layout(bargap=0.05, yaxis_title="기사 수")
fig.show()

print("--- 300자 미만 샘플: 최소 길이 기준 판단용")
show_samples(news[news["body_len"] < 300], 10, seed=1)
for n in (5_000, 10_000, 20_000):
    print(f"본문 {n:,}자 초과: {(news['body_len'] > n).sum():,}건")
print("\n--- 5,000~10,000자 샘플: 정상 장문 기사와 전문·녹취가 섞이는 구간인지 확인")
show_samples(news[news["body_len"].between(5_000, 10_000)], 10, seed=4, width=100)
print("\n--- 가장 긴 기사 10건: 청크를 과도하게 만드는 문서인지 확인")
display(news.nlargest(10, "body_len")[["body_len", "source", "title"]])

count    76344.0
mean      1208.0
std       1181.0
min          0.0
5%         360.0
10%        475.0
25%        660.0
50%        947.0
75%       1416.0
90%       2085.0
95%       2710.0
99%       5301.0
max      53701.0
Name: body_len, dtype: float64
본문 100자 미만: 515건 (0.7%)
본문 200자 미만: 1,376건 (1.8%)
본문 300자 미만: 2,734건 (3.6%)
본문 500자 미만: 8,827건 (11.6%)


--- 300자 미만 샘플: 최소 길이 기준 판단용
[213자] 연합뉴스 | 청주페이 인센티브 연말까지 10% 유지
[0자] 한국경제 | 조폐공사, 전통시장 온누리상품권 활성화 지원
[272자] 이데일리 | [포토]이태원, '헤이, 이태원' 거리전시
[224자] 전주MBC | 4개 대학 동아리연합회, 전통시장 활성화 참여
[176자] 연합뉴스 | [제천소식] 지역화폐 모아 할인율 8%→10% 상향
[121자] 강원일보 | [포토뉴스]인제군 농협들 골목상권 살리기 캠페인
[135자] 연합뉴스 | 정부 내년 지역화폐 예산 전액 삭감
[289자] 뉴스1 | KCC건설‧삼양건설 청주시 신청사 공사기간 주변 식당 이용
[194자] 뉴스1 | 1인당 최대 60만원 지원…'중동 쇼크'에 26.2조 추경
[295자] 노컷뉴스 | 세종시, 지역화폐 캐시백 13%로 상향
본문 5,000자 초과: 864건
본문 10,000자 초과: 205건
본문 20,000자 초과: 34건

--- 5,000~10,000자 샘플: 정상 장문 기사와 전문·녹취가 섞이는 구간인지 확인
[6,346자] 대구MBC | [들어보니] 벼랑 끝 지역화폐 | 빅벙커
[5,335자] 아시아경제 | 용산구 '오후 4시 민원 비우기' 추진...민원처리 단축률 10％↑
[7,850자] 노컷뉴스 | 이범석 청주시장 가시적 성과 창출위한 역동적 시정
[7,214자] 오마이뉴스 | 갈 곳도 없고, 할 것도 없어요... 그래서 위험해요 [월간 옥이네]
[6,941자] 노컷뉴스 | '3선' 이병선 속초시장 민생회복지원금 7월 지급, 미래 100년 준비
[8,777자] YTN | 4·10 총선 D-2...임종석 전 비서실장에게 듣는다
[6,371자] 오마이뉴스 | 김성제 의왕시장 전국에서 가장 살기 좋은 도시 의왕 실현에 총력
[5,420자] 미디어오늘 | 내년도 예산에 거대양당 졸속협상 지역구 환심 급급
[7,069자] 디지털타임스 | [전문]서산 유세서 김수현·박원순 소환한 윤석열 집값 폭등, 실책 아닌 의도적
[5,282

,body_len,source,title
916,53701,SBS,[풀영상] '송이대첩 2탄' 송영길 vs 이준석 100분 당대표 토론배틀
46712,45771,JTBC,이재명 내란 국민의힘 누군가도 동조...책임 있으면 정치인도 처벌해야
11399,42965,노컷뉴스,[민주당 당대표 토론회 전문] 내가 최고 적임자
50648,41805,한국일보,"박찬대, 내란종식특별법…논란은 있지만 민주주의를 더 공고히 할 거라 생각해 [이슈..."
45881,30105,KBS,"[사사건건] 영남으로 달려간 세 후보, 왜?"
46698,29014,KBS,[사사건건] 4당 대선 토론…각 당 전략과 쟁점은?
46268,29006,JTBC,"[전문] 신동욱 “이재명, 총기 허용국가도 아닌데 왜 '방탄유리' 세우나…국민에 대..."
36078,28457,JTBC,[전문]이한주 금투세 논란에 다른 일 못해...폐지하고 추후 정비하는 것이 낫다
41169,28341,MBC,[맞수다] 해산 의도 없는데 군 투입? 따뜻한 아이스 아메리카노 같은 소리
30251,27766,머니투데이,내가 뽑으면 되는겨 예측 불가 충청 표심…여야 모두 목표는 같다


## 4. 품질
- 광고·사진·인사·정기 편성 기사 표지, 보일러플레이트 비중

In [5]:
# 제목 표지는 indexer/quality.py 의 TITLE_MARKERS (괄호가 닫히는 위치까지 맞춰 연재명 오탐 방지)
BODY_MARKERS = {
    "저작권 문구": r"무단\s*전재|재배포\s*금지|Copyright|ⓒ|©",
    "AI 학습 금지 문구": r"AI\s*학습\s*(?:이용)?\s*금지",
    "기자 이메일": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "특수 줄바꿈 문자(LS·PS)": "[\u2028\u2029]",
}
title_hits = {k: news["title"].str.contains(p, regex=True) for k, (p, _) in TITLE_MARKERS.items()}
rows = [("제목", k, m.sum(), m.mean()) for k, m in title_hits.items()]
body_hits = {k: news["text"].str.contains(p, regex=True) for k, p in BODY_MARKERS.items()}
rows += [("본문", k, m.sum(), m.mean()) for k, m in body_hits.items()]
markers = pd.DataFrame(rows, columns=["위치", "표지", "건수", "비율"])
markers["비율"] = markers["비율"].map("{:.1%}".format)
display(markers)

# 제외 후보: 광고, 사진·영상, 인사·부고·게시판, 전문·녹취, 정기 편성
news["marker_excluded"] = pd.concat(title_hits.values(), axis=1).any(axis=1)
for k, m in title_hits.items():
    print(f"--- {k} 예시")
    print(news.loc[m, "title"].head(5).to_string() if m.any() else "    (없음)")

,위치,표지,건수,비율
0,제목,포토·사진·영상,230,0.3%
1,제목,광고·협찬,0,0.0%
2,제목,인사·부고·동정·게시판,83,0.1%
3,제목,전문·풀영상·녹취,66,0.1%
4,제목,정기 편성 (일정·신문 요약),223,0.3%
5,본문,저작권 문구,211,0.3%
6,본문,AI 학습 금지 문구,93,0.1%
7,본문,기자 이메일,0,0.0%
8,본문,특수 줄바꿈 문자(LS·PS),1,0.0%


--- 포토·사진·영상 예시
5        [영상]절박한 이재명 본선채비 '잰걸음'…원팀 호소하며 '몸낮추기'
329     [영상]이재명, 퇴임 기자회견…공정한 나라로 반드시 보답할 것(종합)
392          [영상]지사직 끝낸 李, 오늘 文 면담…본격 대선행보 과제는
1172               [포토] 전통시장 상인과 대화하는 오세훈 서울시장
1173                    [포토] 전통시장 찾아간 오세훈 서울시장
--- 광고·협찬 예시
    (없음)
--- 인사·부고·동정·게시판 예시
404         [게시판] KB국민카드, 포항시와 빅데이터 업무협약
1140        [게시판] 롯데호텔, 전통시장 활성화 프로젝트 진행
3181    [게시판] 부산경제진흥원, 전통시장 활성화로 중기부 장관상
4203                            [인사] 부산시
5336              [동정]원주 횡성 2022년 1월 26일
--- 전문·풀영상·녹취 예시
307                    [전문] 이재명 세계가 부러워하는 선도국가 대한민국 만들겠다
324                       이재명, 지사직 사퇴 공정한 나라로 보답하겠다 [전문]
916             [풀영상] '송이대첩 2탄' 송영길 vs 이준석 100분 당대표 토론배틀
2004                    이재명, 전국민 지원금 철회 방식·대상 고집않겠다 [전문]
2015    [전문] 한 발 뺀 이재명, ‘전국민 재난지원금’ 철회…“국민의힘도 반대하지 않을 것”
--- 정기 편성 (일정·신문 요약) 예시
819          머니투데이 오프라인 헤드라인-2일자
921      [오늘의 주요일정]사회(11월4일 목요일)
1089          11월 5일 '뉴스 9' 헤드라인
1492    [오늘의 주요일정] 정치·정부(15일, 월)
2160        머니투데이 오프라인 헤드라인-22일자


## 5. 서울 비중
- 통신사 기사의 첫머리 발신지 표기 `(서울=연합뉴스)` 는 기사 내용과 무관하므로 제거한 뒤 판정한다
- '서울'이 들어간 고유명사(서울경제, 서울신문, 서울대 등)와 중복 지명(중구·강서구)도 제외한다
- 키워드 단순 매칭이라 근사치다. 오판은 샘플을 읽어 확인한다

In [6]:
LEAD_CHARS = CONFIG.lead_chars  # 제목 + 본문 앞부분을 '리드'로 본다
DATELINE, SEOUL = DATELINE_PATTERN, SEOUL_PATTERN

lead_raw = news["title"] + " " + news["text"].str.slice(0, LEAD_CHARS)
lead = lead_raw.str.replace(DATELINE, " ", regex=True)
news["seoul_lead"] = lead.str.contains(SEOUL, regex=True)
print(f"서울 언급 비율  발신지 표기 포함: {lead_raw.str.contains(SEOUL, regex=True).mean():.1%}"
      f"  →  제거 후: {news['seoul_lead'].mean():.1%}")
display(news.groupby("file_group")["seoul_lead"].mean().map("{:.1%}".format).rename("서울 언급 비율").to_frame())

REGIONS = {"서울": SEOUL_PATTERN, **PROVINCE_PATTERNS}
region_hits = {r: lead.str.contains(p, regex=True) for r, p in REGIONS.items()}
regions = pd.Series({r: m.mean() for r, m in region_hits.items()})
fig = px.bar(regions.rename_axis("지역").reset_index(name="비율"), x="비율", y="지역", orientation="h",
             title=f"제목·리드 {LEAD_CHARS}자 내 광역 지자체 언급 비율",
             color_discrete_sequence=SERIES[:1])
fig.update_yaxes(categoryorder="total ascending", title=None)
fig.update_xaxes(tickformat=".0%")
fig.update_layout(bargap=0.35, height=560)
fig.show()

서울 언급 비율  발신지 표기 포함: 13.8%  →  제거 후: 13.4%


,서울 언급 비율
file_group,
소상공인,19.2%
소상공인 + 지역화폐,6.1%
지역화폐,7.4%


## 6. 범위 분류 — 서울 언급 / 타 지역만 언급 / 지역 언급 없음
- 범위 규칙(정책은 전국, 지자체 사업은 서울)에 따라 '타 지역만 언급'이 제외 후보다

In [7]:
# 범위 규칙: 정책은 전국 단위, 지자체 사업은 서울. 따라서 '서울 언급'뿐 아니라
# 특정 지역이 없는 전국 단위 기사도 범위 안이다. 타 지역만 언급한 기사가 제외 후보다.
# 광역 지자체명만으로는 '하남시', '청주시' 같은 기초 지자체 기사를 놓친다 (첫 실행 샘플에서 확인).
# 'OO시는', 'OO군이'처럼 지자체가 주어로 나오는 형태를 근사로 잡고, 잡힌 이름을 직접 확인한다.
CITY = CITY_PATTERN  # 'OO시는', 'OO특례시가' 등. NOT_PLACE 로 대도시·신도시 등 제외
city_names = lead.str.findall(CITY).map(lambda names: [n for n in names if n not in NOT_PLACE])
print("--- 탐지된 기초 지자체 상위 30개: 지명이 아닌 단어가 섞였는지 확인 (섞였으면 NOT_PLACE 에 추가)")
print(city_names.explode().value_counts().head(30).to_string())

other_region = pd.concat([m for r, m in region_hits.items() if r != "서울"], axis=1).any(axis=1)
other_region |= city_names.map(bool)
news["scope"] = "지역 언급 없음"
news.loc[other_region, "scope"] = "타 지역만 언급"
news.loc[news["seoul_lead"], "scope"] = "서울 언급"
display(pd.crosstab(news["scope"], news["file_group"], margins=True, margins_name="합계"))

print("--- '지역 언급 없음' 샘플: 전국 정책 기사인지, 지역 기사인데 지명이 리드 밖에 있는지 확인")
show_samples(news[news["scope"] == "지역 언급 없음"], 10, seed=2)

--- 탐지된 기초 지자체 상위 30개: 지명이 아닌 단어가 섞였는지 확인 (섞였으면 NOT_PLACE 에 추가)
부산    2888
광주    1595
대전    1481
대구    1314
광명    1254
용인    1201
인천    1200
수원    1080
파주     980
익산     954
세종     900
시흥     819
화성     766
성남     736
안양     715
포항     702
전주     601
부천     544
고양     528
원주     497
청주     489
울산     469
강릉     464
이천     462
오산     429
하남     399
춘천     391
군산     376
부여     371
양산     365


file_group,소상공인,소상공인 + 지역화폐,지역화폐,합계
scope,,,,
서울 언급,7565,184,2504,10253
지역 언급 없음,9073,562,10606,20241
타 지역만 언급,22819,2247,20784,45850
합계,39457,2993,33894,76344


--- '지역 언급 없음' 샘플: 전국 정책 기사인지, 지역 기사인데 지명이 리드 밖에 있는지 확인
[1,779자] 한겨레 | 국감때 되면 ‘집단 발병’하는 기업인들…이번엔 플랫폼 기업
[10,068자] 오마이뉴스 | [전문] 한병도 지방선거때 개헌 제안, 5.18정신 헌법에 수록하자
[601자] JTBC | 줄어든 예산안 야당 단독 처리…여당 보복성 예산 감액
[663자] 뉴스1 | 번영지원금부터 소원성취금까지…당근, 지역 상권 활성화 캠페인
[839자] SBS | 신차보다 큰 '중고차 시장'…현대차, 사업에 첫발 들였다
[1,096자] 한국경제 | 정부案 내년 604조…'이재명 예산' 반영 놓고 與野 정면충돌 예고
[785자] 매일경제 | '이재명표 지역화폐' 국비지원 대폭 늘린다
[1,285자] 부산일보 | 윤 대통령 체포 이후 민주당 행보는…이재명 경제 챙기기 행보로 대선 준비 강화할 듯
[1,906자] 조선일보 | 文뉴딜·李지역화폐 깎고, ‘윤석열표 예산’ 늘린다
[3,711자] 뉴스1 | 국감 첫날 '파행·정회'…'文조사 vs 尹순방·IRA' 충돌(종합2보)


## 7. 주제 관련성
- 제목·리드에 주제어가 없는 기사는 키워드가 본문에만 스친 주제 무관 기사일 가능성이 높다

In [8]:
# 키워드가 본문 어딘가에만 스쳐 나오는 기사(정치·경제 일반)를 걸러내기 위한 관련성 판정어 후보
TOPIC_TERMS = "|".join(map(re.escape, CONFIG.topic_terms))
news["topic_lead"] = lead.str.contains(TOPIC_TERMS, regex=True)
print(f"제목·리드에 주제어가 있는 기사: {news['topic_lead'].mean():.1%}")
display(news.groupby("file_group")["topic_lead"].mean().map("{:.1%}".format).rename("주제어 포함 비율").to_frame())

print("--- 주제어가 없는 기사 샘플: 주제와 무관한지 확인")
show_samples(news[~news["topic_lead"]], 10, seed=3)

제목·리드에 주제어가 있는 기사: 67.7%


,주제어 포함 비율
file_group,
소상공인,72.5%
소상공인 + 지역화폐,79.8%
지역화폐,61.1%


--- 주제어가 없는 기사 샘플: 주제와 무관한지 확인
[670자] 연합뉴스 | 제천에 지은 충북도자치연수원 개원…연간 8천명 교육
[2,522자] 비즈니스워치 | [모빌리티 新3강]①'카카오 택시' 독주 흔들리나
[774자] 헤럴드경제 | 관광 빅데이터 우수 활용사례 찾는다…‘데이터랩 경진대회’ 개최
[725자] 아시아경제 | 의정부시, 경민대학로 더드림 재생사업 ‘경민광장 개장식’ 열려
[1,498자] 스포츠경향 | 김동연 경기지사 “숙성된 대안과 뜬금없는 대안은 다르다”···‘김포 서울 편입’ 재차 비판
[1,169자] 뉴시스 | 서울 중구, 'DDP 일대 지구단위계획 재정비' 주민 의견 모은다
[1,501자] 아이뉴스24 | 尹 이제 내수 활성화…4월부터 전국 대규모 이벤트 쏟아진다
[1,871자] 뉴시스 | 與 尹정부 첫 예산안 통과…경제위기 극복 마중물(종합)
[971자] 아이뉴스24 | 민주 오늘 '이상민 해임안' 처리…金의장, 본회의 열어야
[2,089자] SBS Biz | [직설] '적극 재정' 강조하던 민주당, 예산 삭감…왜?


## 8. 중복 현황
- 같은 URL, 본문 완전 일치, 괄호 머리말을 뗀 제목 일치 → `indexer/dedup.py` 기준 근거

In [9]:
nonempty = news["text"] != ""
news["norm_title"] = (news["title"].str.replace(r"[\[【(][^\]】)]*[\]】)]", "", regex=True)
                      .str.replace(r"\W+", "", regex=True))
titled = news["norm_title"] != ""
dup = pd.DataFrame({
    "관련 문서 수": {
        "같은 URL (발행일만 다름)": news["url"].duplicated(keep=False).sum(),
        "본문 완전 일치": news.loc[nonempty, "text"].duplicated(keep=False).sum(),
        "정규화 제목 일치": news.loc[titled, "norm_title"].duplicated(keep=False).sum(),
    },
    "대표 1건 외 제거 대상": {
        "같은 URL (발행일만 다름)": news["url"].duplicated().sum(),
        "본문 완전 일치": news.loc[nonempty, "text"].duplicated().sum(),
        "정규화 제목 일치": news.loc[titled, "norm_title"].duplicated().sum(),
    },
})
display(dup)

groups = news[titled].groupby("norm_title").filter(lambda g: len(g) > 1)
top = groups["norm_title"].value_counts().head(5).index
display(groups[groups["norm_title"].isin(top)][["norm_title", "published_at", "source", "body_len", "title"]]
        .sort_values(["norm_title", "published_at"]))

,관련 문서 수,대표 1건 외 제거 대상
같은 URL (발행일만 다름),0,0
본문 완전 일치,1627,1018
정규화 제목 일치,5387,3131


,norm_title,published_at,source,body_len,title
5949,경기남부,2022-02-16T06:00:00,뉴시스,1079,[오늘의 주요일정]경기남부(2월16일 수요일)
18972,경기남부,2023-02-15T05:00:00,뉴시스,911,[오늘의 주요일정]경기남부(2월15일 수요일)
19126,경기남부,2023-02-22T05:00:00,뉴시스,1356,[오늘의 주요일정]경기남부(2월22일 수요일)
21656,경기남부,2023-06-14T05:01:00,뉴시스,423,[오늘의 주요일정]경기남부(6월14일 수요일)
25504,경기남부,2023-11-08T05:00:00,뉴시스,2338,[오늘의 주요일정] 경기남부(11월8일 수요일)
...,...,...,...,...,...
70399,작년폐업사업자97만개폐업률86,2026-06-30T14:20:00,연합뉴스,250,작년 폐업 사업자 97만개·폐업률 8.6%
70400,작년폐업사업자97만개폐업률86,2026-06-30T14:20:00,연합뉴스,250,작년 폐업 사업자 97만개·폐업률 8.6%
70401,작년폐업사업자97만개폐업률86,2026-06-30T14:20:00,연합뉴스,250,작년 폐업 사업자 97만개·폐업률 8.6%
70402,작년폐업사업자97만개폐업률86,2026-06-30T14:20:00,연합뉴스,250,작년 폐업 사업자 97만개·폐업률 8.6%


## 9. 내용 — 샘플 직접 읽기
- 구조와 서술 방식, 제거할 반복 문구, 핵심 내용의 위치를 기록한다

In [10]:
SAMPLE_PER_GROUP = 5
for group, g in news.groupby("file_group"):
    print(f"===== {group} =====")
    show_samples(g, SAMPLE_PER_GROUP, seed=42, width=300)

===== 소상공인 =====
[3,047자] 파이낸셜뉴스 | 유정복 ‘제물포 르네상스’ 임기 내 마스터플랜 수립… 원도심 살릴것 [로컬 포커스 자치단체장을 만나다]
[707자] 뉴스1 | 션이 소상공인을 응원합니다…동행축제 홍보영상 120만뷰 돌파
[1,052자] 뉴시스 | 울산형 일상회복 희망 지원금, 골목상권 회복 '효과'
[700자] 대전일보 | [4·10 총선 레이더] 장종태 도마·변동 재개발, 생활인프라 확충 약속
[1,249자] 파이낸셜뉴스 | 광주광역시 서구, '찾아가는 내 곁에 구청장실' 운영 호평
===== 소상공인 + 지역화폐 =====
[2,288자] 프레시안 | 정의당 탐나는전 활성화 정책 흔들림 없이 추진돼야
[685자] 경기일보 | 안산시, 연매출 10억원 초과 295곳 지역화폐 가맹점 취소
[596자] 한국경제 | 골목상권 살리기 해법은 '앱'…하남-신한 손잡았다
[850자] SBS | 이재명 윤, 전국민 지원금 협의하자…윤석열, 김종인 출판기념회 참석
[1,461자] 헤럴드경제 | 野 ‘지역사랑상품권 의무화법’ 추진...예산 ‘0원’ 편성에 맞불
===== 지역화폐 =====
[1,392자] SBS Biz | [이슈앤 직설] 정부, 초과 세수 활용 방안…‘12.7조+α’ 민생 대책
[702자] 뉴스1 | 이천시, ‘가정애(愛) 달’ 지역화폐 10% 캐시백…1인당 최대 10만원
[2,003자] 조선비즈 | [지역화폐 현주소]② 의정부시, 876억 뿌렸더니 학원비로 218억 펑펑
[1,702자] 한겨레 | 대통령이 걷어찬 중재안…협상 원점으로, 예산안 연말까지 표류
[603자] 머니S | 경기도, 31개 시·군 부단체장과 추석 연휴 도민 안전 대책 논의


## 10. 부족 자료 탐색 — 제공되지 않은 가이드 키워드의 본문 등장 건수
- 등장이 적은 항목은 직접 수집(보도자료·서울시 자료·보고서·법령)으로 보강하거나 답변 불가 문항으로 쓴다

In [11]:
GUIDE_TERMS = {
    "소상공인 정책자금": r"정책\s*자금",
    "상권활성화구역": r"상권\s*활성화\s*구역",
    "지역사랑상품권": r"지역사랑\s*상품권|서울사랑\s*상품권",
    "디지털 전환": r"디지털\s*대?전환",
    "상가 공실": r"공실",
    "상권 임대료": r"임대료",
    "젠트리피케이션": r"젠트리피케이션|둥지\s*내몰림",
    "생활인구": r"생활\s*인구",
    "카드·가맹점 수수료": r"(?:카드|가맹점)\s*수수료",
    "소상공인시장진흥공단": r"소상공인시장진흥공단|소진공",
}
in_scope = news["scope"].isin(["서울 언급", "지역 언급 없음"])
coverage = pd.DataFrame({
    "전체": {k: news["text"].str.contains(p, regex=True).sum() for k, p in GUIDE_TERMS.items()},
    "범위 내 (서울+지역 언급 없음)": {k: news.loc[in_scope, "text"].str.contains(p, regex=True).sum() for k, p in GUIDE_TERMS.items()},
    "서울 언급": {k: news.loc[news["seoul_lead"], "text"].str.contains(p, regex=True).sum() for k, p in GUIDE_TERMS.items()},
})
display(coverage)

fig = px.bar(coverage["범위 내 (서울+지역 언급 없음)"].rename_axis("항목").reset_index(name="기사 수"),
             x="기사 수", y="항목", orientation="h", title="범위 내 기사 중 항목별 등장 건수",
             color_discrete_sequence=SERIES[:1])
fig.update_yaxes(categoryorder="total ascending", title=None)
fig.update_layout(bargap=0.35)
fig.show()

,전체,범위 내 (서울+지역 언급 없음),서울 언급
소상공인 정책자금,1320,707,229
상권활성화구역,307,65,20
지역사랑상품권,7059,4159,955
디지털 전환,1719,904,289
상가 공실,1017,348,193
상권 임대료,1683,979,417
젠트리피케이션,171,105,82
생활인구,534,117,65
카드·가맹점 수수료,760,174,78
소상공인시장진흥공단,2321,1235,389


## 11. 지식베이스 후보 규모 — `indexer/quality.py` v1 적용 결과
- 인덱서가 쓸 `assess()` 를 그대로 돌린다. 사유는 판정 순서상 처음 걸린 하나만 센다
- 기간 규칙의 근거가 된 연도별·기간별 표를 함께 남긴다 (v1 = 5년 전체, 비교 실험 v1-2y = 최근 2년)

In [12]:
from indexer.dedup import content_hash
from indexer.parse import clean_text

# prepare 와 같게 정제 본문(저작권 문구 제거 등)으로 판정한다. 1~10번 탐색은 원문 기준이다.
news["clean"] = news["text"].map(clean_text)
decisions = [
    assess(QualityInput(doc_id=r.doc_id, doc_type=DocType.NEWS, title=r.title, text=r.clean,
                        source=r.source, published_at=r.published_at, extra=r.extra), CONFIG)
    for r in news.itertuples(index=False)
]
news["q_reason"] = [d.reason.value if d.excluded else "유지" for d in decisions]
news["q_scope"] = [d.scope.value if d.scope else None for d in decisions]

ORDER = ["too_short", "too_long", "low_information", "promotional", "transcript",
         "off_topic", "outside_seoul", "out_of_period", "유지"]
display(news["q_reason"].value_counts().reindex(ORDER, fill_value=0).rename("건수").to_frame())

# 남은 문서 안에서만 중복을 판정해야, 앞 단계에서 빠진 원본 때문에 사본까지 사라지지 않는다.
# 완전 중복은 indexer/dedup.py 와 같게 공백·기호를 뗀 정제 본문의 해시로 판정한다.
keep = news["q_reason"] == "유지"
dupes = news.loc[keep, "clean"].map(content_hash).duplicated()
keep[dupes[dupes].index] = False
print(f"품질 필터 통과 {int((news['q_reason'] == '유지').sum()):,}건 → 완전 중복 제거 후 {int(keep.sum()):,}건")
candidates = news[keep]
display(candidates["file_group"].value_counts().rename("최종 후보").to_frame())
display(candidates["q_scope"].value_counts().rename("최종 후보").to_frame())

# 기간 규칙의 근거: 기간 규칙만 빼고 모두 통과한 기사의 연도별·기간별 건수
pool = news[news["q_reason"].isin(["유지", "out_of_period"])]
display(pool.groupby(pool["published"].dt.year).size().rename("기간 규칙 전 후보").to_frame())
latest = news["published"].max()
display(pd.DataFrame(
    {f"최근 {y}년": [int((pool["published"] > latest - pd.DateOffset(years=y)).sum())] for y in (1, 2, 3, 5)},
    index=["기간 규칙 전 후보"],
))

CHUNK_CHARS = 800
total_chars = int(candidates["clean"].str.len().sum())
print(f"최종 후보 본문 합계 {total_chars:,}자 → 청크 약 {total_chars // CHUNK_CHARS:,}개 (청크 {CHUNK_CHARS}자 가정)")
print("※ 재배포본(본문 유사) 처리는 아직 반영되지 않았다. 실제 지식베이스는 이보다 작아진다.")

,건수
q_reason,
too_short,551
too_long,199
low_information,484
promotional,0
transcript,24
off_topic,24180
outside_seoul,32283
out_of_period,0
유지,18623


품질 필터 통과 18,623건 → 완전 중복 제거 후 18,230건


,최종 후보
file_group,
소상공인,11426
지역화폐,6252
소상공인 + 지역화폐,552


,최종 후보
q_scope,
national,11363
seoul,6867


,기간 규칙 전 후보
published,
2021,1609
2022,3232
2023,2317
2024,2873
2025,4950
2026,3642


,최근 1년,최근 2년,최근 3년,최근 5년
기간 규칙 전 후보,4418,9301,12146,18617


최종 후보 본문 합계 22,518,662자 → 청크 약 28,148개 (청크 800자 가정)
※ 재배포본(본문 유사) 처리는 아직 반영되지 않았다. 실제 지식베이스는 이보다 작아진다.


## 12. 재배포본 — 중복 처리 임계값 (`indexer/dedup.py`)
- 11번 최종 후보끼리 발행일 3일 이내 쌍의 본문 유사도(문자 5-gram Jaccard)를 본다
- 구간별 표본을 읽고 **같은 기사의 재배포**와 **관점이 다른 별개 기사**가 갈리는 지점을 임계값으로 정한다
- 관점이 다른 기사까지 지우면 상충 근거 제시가 불가능해지므로 임계값을 과도하게 낮추지 않는다 (가이드 4-4)

In [13]:
from indexer.dedup import DedupInput, count_duplicates, get_dedup_config, similar_pairs

DEDUP = get_dedup_config("v1")
docs = [DedupInput(r.doc_id, DocType.NEWS, r.clean, r.published_at) for r in candidates.itertuples(index=False)]
pairs = pd.DataFrame(similar_pairs(docs, DEDUP, min_similarity=0.3), columns=["a", "b", "similarity"])
print(f"비교 대상 {len(docs):,}건 / 유사도 0.3 이상 쌍 {len(pairs):,}개 (발행일 차이 {DEDUP.max_day_gap}일 이내)")

fig = px.histogram(pairs, x="similarity", nbins=35, range_x=[0.3, 1.0], title="문서 쌍 유사도 분포 (문자 5-gram Jaccard)",
                   labels={"similarity": "유사도"}, color_discrete_sequence=SERIES[:1])
fig.update_layout(bargap=0.05, yaxis_title="문서 쌍 수")
fig.show()

info = candidates.set_index("doc_id")
for lo, hi in [(0.3, 0.5), (0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 0.9), (0.9, 1.01)]:
    band = pairs[(pairs["similarity"] >= lo) & (pairs["similarity"] < hi)]
    print(f"\n--- 유사도 {lo:.1f}~{min(hi, 1.0):.1f}: {len(band):,}쌍 — 같은 기사의 재배포인가, 별개 기사인가?")
    for _, pair in band.sample(min(5, len(band)), random_state=5).iterrows():
        a, b = info.loc[pair["a"]], info.loc[pair["b"]]
        print(f"[{pair['similarity']:.2f}] {a['published_at'][:10]} {a['source']} | {a['title']}")
        print(f"       {b['published_at'][:10]} {b['source']} | {b['title']}")

rows = []
for threshold in (0.5, 0.6, 0.7, 0.8, 0.9):
    removed = count_duplicates(info.index, pairs[["a", "b", "similarity"]].itertuples(index=False, name=None), threshold)
    rows.append((threshold, removed, len(docs) - removed))
display(pd.DataFrame(rows, columns=["임계값", "제거되는 재배포본", "남는 뉴스"]))
print("최종 건수는 python -m indexer prepare 출력이 기준이다 (완전 중복과 재배포본을 함께 묶어 계산).")

비교 대상 18,230건 / 유사도 0.3 이상 쌍 12,540개 (발행일 차이 3일 이내)



--- 유사도 0.3~0.5: 6,877쌍 — 같은 기사의 재배포인가, 별개 기사인가?
[0.34] 2025-07-25 연합뉴스 | '케미스트릿 강남역' 상권협의체 출범…글로컬상권 조성 박차
       2025-07-25 뉴시스 | 서초구, '케미스트릿 강남역 상권협의체' 출범
[0.31] 2024-09-26 연합뉴스 | 폐업 개식용농장에 마리당 최대 60만원…농가당 2억4천만원 내외
       2024-09-26 한국경제TV | 마리당 최대 60만 원 지원, 개 식용 종식 로드맵 발표
[0.45] 2023-05-17 이데일리 | 중소기업계 “내수·고용·수출 5% 이상 늘릴 것”
       2023-05-17 아이뉴스24 | 중기중앙회, '민생활력 온도 +5℃ 캠페인' 출범
[0.40] 2022-02-25 동아일보 | 이재명, 자영업자 지지 선언에 “기쁨보다 송구함 앞서”
       2022-02-25 이데일리 | 이재명 전국민 지역화폐, 올여름 가기 전 확대 발행
[0.47] 2025-07-07 아시아경제 | 대·중소기업 함께 민생 살리기 나섰다
       2025-07-07 아이뉴스24 | 중기중앙회-한경협, 민생살리기 업무협약 체결

--- 유사도 0.5~0.6: 2,004쌍 — 같은 기사의 재배포인가, 별개 기사인가?
[0.56] 2025-03-04 디지털데일리 | '은행·카드' 금융데이터가 지역 상권을 살려낼 수 있을까
       2025-03-04 지디넷코리아 | KB국민은행·KB국민카드, 송파구청과 데이터 활용 협약
[0.52] 2025-09-02 이데일리 | 소진공, 소비활성화 캠페인 본격 추진
       2025-09-02 한국경제TV | 소진공, '소비활성화 캠페인'…지역상권 활력
[0.55] 2025-04-23 파이낸셜뉴스 | 골목상권 살리는 글로컬 상권·로컬브랜드 창출 8개팀 선정
       2025-04-23 전자신문 | 중기부, '글로컬 상권·로컬브랜드 창출사업' 8개팀 선정…“지역 골목에 활력 불어넣는다”
[0.52]

,임계값,제거되는 재배포본,남는 뉴스
0,0.5,3173,15057
1,0.6,2334,15896
2,0.7,1614,16616
3,0.8,924,17306
4,0.9,379,17851


최종 건수는 python -m indexer prepare 출력이 기준이다 (완전 중복과 재배포본을 함께 묶어 계산).


## 13. 같은 보도자료 기사 — 제목 유사도 (`indexer/dedup.py` v2)
- 12번에서 매체마다 같은 보도자료를 다시 쓴 기사가 본문 유사도 0.35~0.55 에 머무는 것을 확인했다
- 같은 구간에 관점이 다른 분석 기사도 섞여 있으므로, 본문 유사도(0.3 이상)와 제목 유사도를 함께 본다
- 제목 유사도 구간별 표본을 읽고 **같은 사안을 같은 내용으로 보도한 쌍**이 시작되는 지점을 `title_threshold` 로 정한다

In [14]:
from indexer.dedup import title_similarity

pairs["title_similarity"] = [title_similarity(info.loc[a, "title"], info.loc[b, "title"])
                             for a, b in zip(pairs["a"], pairs["b"])]
mid = pairs[pairs["similarity"] < 0.8]  # 본문만으로는 재배포본으로 보지 않는 구간
print(f"본문 유사도 0.3~0.8 쌍 {len(mid):,}개의 제목 유사도 분포")
fig = px.histogram(mid, x="title_similarity", nbins=40, title="제목 유사도 분포 (본문 유사도 0.3~0.8 쌍)",
                   labels={"title_similarity": "제목 유사도"}, color_discrete_sequence=SERIES[:1])
fig.update_layout(bargap=0.05, yaxis_title="문서 쌍 수")
fig.show()

for lo, hi in [(0.0, 0.2), (0.2, 0.3), (0.3, 0.4), (0.4, 0.5), (0.5, 0.6), (0.6, 1.01)]:
    band = mid[(mid["title_similarity"] >= lo) & (mid["title_similarity"] < hi)]
    print(f"\n--- 제목 유사도 {lo:.1f}~{min(hi, 1.0):.1f}: {len(band):,}쌍 — 같은 내용의 보도인가, 다른 관점의 기사인가?")
    for _, pair in band.sample(min(6, len(band)), random_state=7).iterrows():
        a, b = info.loc[pair["a"]], info.loc[pair["b"]]
        print(f"[본문 {pair['similarity']:.2f} · 제목 {pair['title_similarity']:.2f}] {a['source']} | {a['title']}")
        print(f"{'':22}{b['source']} | {b['title']}")

ids = info.index
edges = lambda mask: pairs.loc[mask, ["a", "b", "similarity"]].itertuples(index=False, name=None)
rows = [("v1: 본문 ≥ 0.8", count_duplicates(ids, edges(pairs["similarity"] >= 0.8), 0))]
for t in (0.3, 0.4, 0.5, 0.6):
    rule = (pairs["similarity"] >= 0.8) | (pairs["title_similarity"] >= t)  # pairs 는 이미 본문 0.3 이상
    removed = count_duplicates(ids, edges(rule), 0)
    rows.append((f"본문 ≥ 0.8 또는 (본문 ≥ 0.3 · 제목 ≥ {t})", removed))
table = pd.DataFrame(rows, columns=["규칙", "제거되는 기사"])
table["남는 뉴스"] = len(docs) - table["제거되는 기사"]
display(table)

본문 유사도 0.3~0.8 쌍 11,423개의 제목 유사도 분포



--- 제목 유사도 0.0~0.2: 3,084쌍 — 같은 내용의 보도인가, 다른 관점의 기사인가?
[본문 0.41 · 제목 0.11] 이코노미스트 | 국힘‧민주, 13조8000억원 추경안 합의…오늘 본회의서 처리
                      연합뉴스 | 13.8조 추경안 합의…지역화폐 4천억 반영·검찰 등 특경비 복원(종합2보)
[본문 0.31 · 제목 0.12] 한국경제 | 李대통령 중동전쟁 비상대응체계…추경은 지역화폐로 지원 [종합]
                      JIBS | 李 추경, 현금보다 지역화폐로...어려운 사람에 더 많이  지급해야
[본문 0.59 · 제목 0.19] 연합뉴스 | 성탄절·연말까지 밀리나…예산안 처리 여전히 먹구름
                      헤럴드경제 | 예산안 처리 계속 먹구름…크리스마스도 넘기나
[본문 0.39 · 제목 0.17] 뉴시스 | 소상공인 자영업자 발언 메모하는 이재명 민주당 대선 후보
                      뉴스1 | 소상공인 농성 현장 찾은 이재명
[본문 0.32 · 제목 0.15] 매일경제 | 정부, 내년도 예산안 728조 제출…5일 예산 국회 개막
                      경기일보 | 728조원 이재명 정부 첫 예산안 심사 돌입…여야, 정면충돌 예고
[본문 0.57 · 제목 0.06] 머니S | 민생 회복 위해 지역화폐 발행 등 추가 경정 편성해야
                      한국경제 | 민주 정부, 추경 편성으로 지역경제 살려내야

--- 제목 유사도 0.2~0.3: 2,196쌍 — 같은 내용의 보도인가, 다른 관점의 기사인가?
[본문 0.69 · 제목 0.23] 이데일리 | 2만원 이상 외식 4번하면 1만원 환급…하루 실적 최대 2회 인정
                      아시아경제 | 2만원 이상 4번 밥 사먹으면 1만원 돌려준다…하루 2번까지 인정
[본문 0.39 · 제목 0.29] 뉴시스 | 대형마트 의무휴업 11년…전통시

,규칙,제거되는 기사,남는 뉴스
0,v1: 본문 ≥ 0.8,924,17306
1,본문 ≥ 0.8 또는 (본문 ≥ 0.3 · 제목 ≥ 0.3),3803,14427
2,본문 ≥ 0.8 또는 (본문 ≥ 0.3 · 제목 ≥ 0.4),3092,15138
3,본문 ≥ 0.8 또는 (본문 ≥ 0.3 · 제목 ≥ 0.5),2575,15655
4,본문 ≥ 0.8 또는 (본문 ≥ 0.3 · 제목 ≥ 0.6),2053,16177


## 정리 (`indexer/quality.py` v1 에 반영됨)
- 최소·최대 길이: 100자 / 10,000자 (3번)
- 제목 표지 제외: 사진·영상, 광고, 인사·부고·게시판, 전문·녹취, 정기 편성 (4번)
- 범위: 서울 언급 + 지역 언급 없음(전국 단위) 유지, 타 지역만 언급은 제외 (5·6번)
- 주제 관련성: 제목·리드 300자에 주제어 (7번)
- 기간: 제공 기간 5년 전체 (11번). 최근 2년(v1-2y)은 비교 실험으로 남긴다
- 중복 처리 기준 (`dedup.py` v2): 발행일 3일 이내 + (본문 ≥ 0.8 또는 본문 ≥ 0.3 · 제목 유사도 ≥ 0.3) (12·13번)
- 직접 수집 보강 대상 (10번): 상권활성화구역·생활인구·카드수수료·젠트리피케이션